# TLM Fine-tuning: Cree–English XLM-R

Translation Language Modeling (Lample & Conneau, 2019).  
Sentence pairs are concatenated, tokens are masked from both sides, and the model
learns to fill blanks by attending to both languages.  
Input: `sentences.txt` with lines `cree ||| english`.

In [ ]:
# Install / upgrade if needed (comment out after first run)
# !pip install -q transformers[torch] accelerate

## 1 · Config

In [ ]:
from dataclasses import dataclass

@dataclass
class TLMConfig:
    sentences_path:  str   = "sentences.txt"   # path to the uploaded file
    model_name:      str   = "xlm-roberta-base"
    output_dir:      str   = "tlm_model"
    max_length:      int   = 256    # tokens per concatenated pair; XLM-R max = 512
    mlm_probability: float = 0.15
    batch_size:      int   = 16
    grad_accum:      int   = 2
    epochs:          int   = 3
    learning_rate:   float = 2e-5
    warmup_ratio:    float = 0.06
    weight_decay:    float = 0.01
    dev_ratio:       float = 0.05
    seed:            int   = 42

cfg = TLMConfig()
print(cfg)

## 2 · Load sentence pairs

In [ ]:
def load_pairs(path: str) -> list[tuple[str, str]]:
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "|||" not in line:
                continue
            cree, en = line.split("|||", 1)
            pairs.append((cree.strip(), en.strip()))
    return pairs

all_pairs = load_pairs(cfg.sentences_path)
print(f"Loaded {len(all_pairs):,} sentence pairs")
print("Example:", all_pairs[0])

## 3 · Train / dev split

In [ ]:
import random

rng = random.Random(cfg.seed)
pairs = list(all_pairs)
rng.shuffle(pairs)

split = max(1, int(len(pairs) * (1 - cfg.dev_ratio)))
train_pairs = pairs[:split]
dev_pairs   = pairs[split:]

print(f"Train: {len(train_pairs):,}  |  Dev: {len(dev_pairs):,}")

## 4 · Device detection

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    precision_kwargs = {"fp16": True, "bf16": False}
elif torch.backends.mps.is_available():
    device = "mps"
    precision_kwargs = {"fp16": False, "bf16": False}
else:
    device = "cpu"
    precision_kwargs = {"fp16": False, "bf16": False}

print(f"Device : {device}")
print(f"fp16   : {precision_kwargs}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

## 5 · Tokeniser & Dataset

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class TLMDataset(Dataset):
    """
    Each example is encoded as a single sequence:
        <s> cree_tokens </s></s> english_tokens </s>   (XLM-R sentence-pair format)
    Masking is applied by the DataCollator at training time so masks
    are re-sampled each epoch.
    """
    def __init__(self, pairs, tokenizer, max_length):
        self.examples = []
        for src, tgt in pairs:
            enc = tokenizer(
                src, tgt,
                max_length=max_length,
                truncation=True,
                padding=False,
            )
            # token_type_ids are all-zero for XLM-R; drop to avoid DataCollator confusion
            self.examples.append({
                "input_ids":      enc["input_ids"],
                "attention_mask": enc["attention_mask"],
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

print("Tokenising training set …")
train_ds = TLMDataset(train_pairs, tokenizer, cfg.max_length)
print("Tokenising dev set …")
dev_ds   = TLMDataset(dev_pairs,   tokenizer, cfg.max_length)

print(f"Train examples: {len(train_ds):,}")
print(f"Dev   examples: {len(dev_ds):,}")
print(f"Sample lengths: {[len(train_ds[i]['input_ids']) for i in range(3)]}")

## 6 · Model & Collator

In [ ]:
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling

model = AutoModelForMaskedLM.from_pretrained(cfg.model_name)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=cfg.mlm_probability,
)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model parameters: {n_params:.0f}M")

## 7 · Training

In [ ]:
import os
from transformers import Trainer, TrainingArguments

os.makedirs(cfg.output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    report_to="none",
    seed=cfg.seed,
    **precision_kwargs,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
)

trainer.train()

## 8 · Save model

In [ ]:
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"Saved to: {cfg.output_dir}")

## 9 · Quick sanity check

In [ ]:
from transformers import pipeline

fill = pipeline("fill-mask", model=cfg.output_dir, device=0 if device == "cuda" else -1)

# Mask a word in an English sentence and check predictions
test_sent = "They had no clothes, no <mask>."
print(f"Input: {test_sent}")
for r in fill(test_sent)[:5]:
    print(f"  {r['score']:.3f}  {r['token_str']}")